# 01 — Data Collection

**Route:** C81 (Campbell Airport, Grayslake, IL) → KDLH (Duluth International, MN)

This is the first module in the VFR waypoint recommender project. The goal
here is purely `pandas`/data-wrangling: pull real candidate landmarks
(lakes, towers, stadiums, towns, ...) from OpenStreetMap along our route,
and save a clean table for the next notebook to build features on.

Steps:
1. Look up departure/destination coordinates (OurAirports)
2. Compute the direct route and a search corridor around it
3. Query OpenStreetMap (Overpass API) for candidate landmarks in that corridor
4. Enforce the *real* corridor width using great-circle cross-track distance
5. Clean up and save `data/processed/candidates_c81_kdlh.csv`
6. Sanity-check the result on a map


In [ ]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd
from vfr import airports, geo, pipeline

pd.set_option("display.max_columns", None)

## Step 1 — Airport coordinates

`vfr.airports.get_airport` downloads (and caches) OurAirports' free
`airports.csv` and looks up an identifier by either its `ident` or
`local_code` column — needed because `C81` is an FAA local identifier,
not an ICAO code.


In [2]:
dep = airports.get_airport("C81")
dest = airports.get_airport("KDLH")

route_distance_nm = geo.distance_nm(dep["lat"], dep["lon"], dest["lat"], dest["lon"])
route_bearing_deg = geo.bearing_deg(dep["lat"], dep["lon"], dest["lat"], dest["lon"])

print(dep)
print(dest)
print(f"Direct distance: {route_distance_nm:.1f} nm, initial bearing: {route_bearing_deg:.0f} deg")


{'ident': 'KC81', 'name': 'Campbell Airport', 'lat': 42.32460021972656, 'lon': -88.0740966796875, 'municipality': 'Grayslake', 'region': 'US-IL'}
{'ident': 'KDLH', 'name': 'Duluth International Airport', 'lat': 46.841873, 'lon': -92.198746, 'municipality': 'Duluth', 'region': 'US-MN'}
Direct distance: 323.4 nm, initial bearing: 328 deg


## Step 2 — Search corridor

Waypoints need to sit essentially right on the direct C81→KDLH course, so
there are two bands:
- **Preferred: +/-0.25 nm** — practically on the line
- **Fallback: +/-1 nm** — only used later (Notebook 07) to fill gaps where
  the 0.25 nm band has nothing for a long stretch (rural/forest terrain is
  sparse in OSM lakes/towers, and the tight band alone leaves some gaps
  of 20-40 nm on this route)

This notebook queries out to the *fallback* width (2 nm) so both bands
exist in the data, and tags each candidate with which band it falls in.
The actual "only widen where needed" logic is a route-assembly decision
that belongs in Notebook 07, once we have scores to decide what counts as
"needed" — here we're just collecting the candidate pool.

We don't yet know the exact great-circle corridor shape, so we start with
a generous rectangular bounding box around the two endpoints (padded by
the corridor half-width) and query OpenStreetMap within that box. The
*real* corridor constraint (perpendicular distance from the direct route)
gets enforced precisely in Step 4 using `geo.cross_track_distance_nm`.


In [ ]:
# The same bands vfr.pipeline.collect filters with -- read from it, not
# written again here.
PREFERRED_HALF_WIDTH_NM = pipeline.PREFERRED_HALF_WIDTH_NM  # the "essentially on the line" band
FALLBACK_HALF_WIDTH_NM = pipeline.FALLBACK_HALF_WIDTH_NM  # outer bound we collect data for, to fill gaps later

route_start = (dep["lat"], dep["lon"])
route_end = (dest["lat"], dest["lon"])
bbox = geo.corridor_bbox(route_start, route_end, FALLBACK_HALF_WIDTH_NM + 2)
print(f"Bounding box (min_lat, min_lon, max_lat, max_lon): {bbox}")

## Step 3 — Run the collection

From here on the code is `vfr.pipeline.collect` -- the same function the
Airflow DAG and `docker compose run pipeline-processing collect` run, so
this notebook and the pipeline can never write two different candidate
files. It queries OpenStreetMap through the public Overpass API for a
handful of landmark categories inside the bounding box above
(`vfr.osm.query_overpass`, parsed by `parse_overpass_response`), adds the
features of the next step, enforces the corridor, dedupes and drops
unnamed water, then saves the table. The cells below read that table back
and look at each of those steps' results.

In [ ]:
out_path = pipeline.collect("C81", "KDLH")
candidates_df = pd.read_csv(out_path)
candidates_df["tags"] = candidates_df["tags"].apply(json.loads)
print(out_path, candidates_df.shape)
candidates_df.head()

## Step 3b — Rivers, railroads, intersections, wind farms, and FAA navaids/airports

None of these are single-point OSM features the way lakes/towers/stadiums
are, so they get their own resolvers in `vfr.osm` rather than an entry in
`CANDIDATE_SPECS`:
- **Rivers/railroads**: resolved to the point where the route crosses
  them (a pilotage checkpoint for either is "the point you cross it," not
  the feature as a whole).
- **Highway intersections**: not a taggable OSM feature at all -- a node
  shared by 2+ genuinely different *numbered* roads (major highways only:
  motorway/trunk/primary/secondary).
- **Wind farms**: individually-mapped turbines, clustered into one
  farm-level candidate per cluster.

VOR navaids and airports come from the FAA's own NASR data
(`vfr.faa_data`) instead of OSM tags -- the FAA is the authority on its
own navaid and airport network. That data is downloaded once and cached
under `data/raw/faa_nasr/` (28-day publication cycle, much coarser than a
per-route query, so it isn't worth re-fetching every run).

Registered obstacles (the FAA's Digital Obstacle File) are deliberately
*not* collected here, even though `vfr.faa_data.load_obstacles` reads the
same NASR download and notebook 08's terrain/obstacle clearance still
uses it. A sectional draws every obstacle with the same symbol regardless
of what the structure actually is, so there's nothing on the chart to
distinguish one tower from another -- and a tower is a small,
easily-missed target next to what pilotage actually leans on: water
bodies, road intersections, wind farms, towns, and now charted airports.
Excluding them removes towers from consideration entirely rather than
teaching the model to score them low, since the model never sees one to
begin with -- same outcome for recommendations, without spending
labeling effort on candidates that couldn't be rated consistently. A
charted airport, by contrast, is among the most unambiguous things on a
sectional, so a field in the corridor (excluding the two route
endpoints) is now collected as a strong checkpoint candidate instead.

In [ ]:
# What the line and cluster resolvers contributed: river and railroad
# crossings, highway intersections and wind farms.
resolved = ["river", "railroad", "intersection", "wind_farm"]
candidates_df[candidates_df["category"].isin(resolved)].groupby("category")[["name", "along_track_nm"]].agg(
    {"name": "count", "along_track_nm": ["min", "max"]}
)

In [ ]:
# And the FAA's own: charted airports in the corridor (the two route
# endpoints excluded) and VOR navaids.
candidates_df[candidates_df["category"].isin(["airport", "vor"])][["name", "category", "along_track_nm", "cross_track_nm"]]

## Step 4 — Enforce the real corridor

For every candidate, compute:
- `cross_track_nm`: perpendicular distance from the direct C81→KDLH course
- `along_track_nm`: distance along the course from C81 to the candidate's
  projection onto the route (this is what Notebook 07 will use to space
  waypoints out)

Keep candidates within the *fallback* band (+/-1 nm) and not too far
behind the departure or past the destination, and flag which ones also
fall inside the tighter *preferred* band (+/-0.25 nm) with
`within_preferred_corridor`.


In [ ]:
# collect places every candidate on the course in one vectorised call
# (vfr.geo.track_distances_nm) and keeps those within the fallback band.
print(candidates_df[["cross_track_nm", "along_track_nm"]].describe())
print(f"Within the preferred band: {int(candidates_df['within_preferred_corridor'].sum())} of {len(candidates_df)}")

## Step 5 — Clean up

Drop exact-duplicate coordinates (the same feature sometimes comes back
as both a way and an overlapping relation), and take a look at what
categories we actually found — this is the sanity check for whether the
Overpass query and corridor filter are doing something reasonable before
we build features on top of it.


In [ ]:
candidates_df["category"].value_counts()

## Step 5b — Cut obvious noise

OpenStreetMap tags an enormous number of farm ponds and drainage features
with the exact same `natural=water` tag as an actual lake — in this
corridor that's ~10,000 water polygons, and 91% of them have no name at
all. `collect` drops unnamed water whatever its size: the label is a
judgment made against the chart, and at 1:500,000 an unnamed lake is one
blue shape among several identical ones, which cannot be confirmed as the
right one. A size floor (first 40,000 m², later 250,000 m²) was tried
first and still left ponds that were undrawn or unidentifiable -- see the
comment in `vfr.pipeline.collect`. A charted name is the identification,
so named water stays whatever its size.

In [ ]:
water = candidates_df[candidates_df["category"].isin(["lake_or_pond", "reservoir"])]
print(f"{len(water)} water candidates, {int(water['name'].isna().sum())} of them unnamed")

## Step 6 — Saved

`collect` saved the table to `data/processed/candidates_c81_kdlh.csv` for
Notebook 02 (feature engineering), written whole beside the old file and
renamed over it. The `tags` column (a dict of raw OSM tags) is a JSON
string there, so it survives a round-trip through CSV.

In [ ]:
n_preferred = int(candidates_df["within_preferred_corridor"].sum())
print(f"{len(candidates_df)} candidates in {out_path.name}: {n_preferred} within +/-{PREFERRED_HALF_WIDTH_NM} nm, "
      f"{len(candidates_df) - n_preferred} more within +/-{FALLBACK_HALF_WIDTH_NM} nm")

## Step 7 — Map sanity check

A quick visual gut check: does the route line look right, and do the
candidate markers look like plausible landmarks along it? Worth eyeballing
a few on a real map (e.g. search the name on Google Maps) before trusting
this data in the next notebook.


In [ ]:
import folium
from vfr.config import VFR_SECTIONAL_MAX_ZOOM, VFR_SECTIONAL_MIN_ZOOM

# The sectional as the planner renders it from the FAA's own GeoTIFFs
# (vfr.charts), reached from the browser this map opens in -- the hosted
# chart service this used to name is gone. Needs `docker compose up -d
# planning-service`.
SECTIONAL_TILES = "http://localhost:8084/api/chart-tile/sec/{z}/{x}/{y}.png"

mid_lat = (dep["lat"] + dest["lat"]) / 2
mid_lon = (dep["lon"] + dest["lon"]) / 2
# VFR Sectional is the default base layer (tiles=None below, then added
# first) since that's the chart a pilot actually plans against;
# OpenStreetMap is added second as a togglable alternate -- switch to it
# via the layer control (top right) for whole-route context. zoom_start
# is pinned to the sectional's own min zoom (8) rather than a wider
# whole-route zoom, since the tile service renders nothing below that --
# starting any wider would just show a blank chart on load.
# minZoom=6 is set explicitly on the map itself so you can still scroll
# out past the sectional layer's own min zoom of 8 to see the whole route
# -- the chart just goes blank below 8, but the polyline/markers are
# vector overlays and stay visible regardless. NOTE: folium.Map's own
# min_zoom= kwarg only takes effect when tiles= is a plain string (it
# gets forwarded into folium's *implicit* default TileLayer); since we
# pass tiles=None and add layers manually below, that kwarg is silently
# dropped -- minZoom (Leaflet's actual camelCase option name, passed
# through Map's **kwargs straight into the JS map options) is what
# actually reaches the Leaflet map object.
m = folium.Map(
    location=[mid_lat, mid_lon],
    zoom_start=VFR_SECTIONAL_MIN_ZOOM,
    minZoom=6,
    tiles=None,
    height="100%",
)
# height="100%" so the saved standalone HTML fills the whole browser tab --
# this is meant to be opened as its own page, not viewed inline in the
# notebook cell output (a plain "100%" collapses to 0 there since VS Code's
# notebook output area doesn't give the map a real height to fill against)
folium.TileLayer(
    tiles=SECTIONAL_TILES,
    attr="FAA Aeronautical Information Services",
    name="VFR Sectional",
    max_zoom=VFR_SECTIONAL_MAX_ZOOM,
    min_zoom=VFR_SECTIONAL_MIN_ZOOM,
).add_to(m)
folium.TileLayer(tiles="OpenStreetMap", name="OpenStreetMap", show=False).add_to(m)
folium.LayerControl().add_to(m)

folium.PolyLine([[dep["lat"], dep["lon"]], [dest["lat"], dest["lon"]]], color="blue", weight=2).add_to(m)
folium.Marker([dep["lat"], dep["lon"]], tooltip=f"Departure: {dep['ident']}", icon=folium.Icon(color="green")).add_to(m)
folium.Marker([dest["lat"], dest["lon"]], tooltip=f"Destination: {dest['ident']}", icon=folium.Icon(color="red")).add_to(m)

for _, r in candidates_df.iterrows():
    color = "orange" if r["within_preferred_corridor"] else "gray"
    folium.CircleMarker(
        [r["lat"], r["lon"]],
        radius=4,
        popup=f"{r['name'] or '(unnamed)'} ({r['category']}, cross-track {r['cross_track_nm']:.2f} nm)",
        color=color,
        fill=True,
    ).add_to(m)

map_path = PROJECT_ROOT / "data" / "processed" / "candidates_map.html"
m.save(str(map_path))
print(f"Map saved to {map_path}")
m